# Colab: prepare data and run a short training test

Select a Colab T4 GPU kernel. Run the settings and training cells to prepare and train. The separate status cell can inspect the same runtime after the active cell stops. The default is a short smoke test. Files in `/content` disappear when the server is removed.


In [ ]:
REPO_URL = "https://github.com/EdwinKam/emg2qwerty.git"
PROJECT_REF = "376923b"  # Published commit; use another published ref if needed.
MAX_STEPS = 10  # Increase for real training.
BATCH_SIZE = 4


In [ ]:
import os, subprocess, sys, time, tarfile, threading, signal, urllib.request
from datetime import datetime
from pathlib import Path
try:
    import google.colab
    import torch
except ImportError as exc:
    raise RuntimeError('Select a hosted Colab GPU kernel first.') from exc
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before continuing.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
root = Path('/content')
repo = root / 'emg2qwerty'
data = root / 'emg2qwerty-data-2021-08'
archive = root / 'emg2qwerty-data-2021-08.tar.gz'
env = root / 'emg2qwerty-env'
mamba = root / 'bin/micromamba'
url = 'https://fb-ctrl-oss.s3.amazonaws.com/emg2qwerty/emg2qwerty-data-2021-08.tar.gz'
status_path = root / 'emg2qwerty-status.json'
def status(stage, state='running', detail=''):
    record = {'stage': stage, 'state': state, 'detail': detail, 'updated': datetime.now().astimezone().isoformat()}
    temporary = status_path.with_suffix('.tmp')
    temporary.write_text(__import__('json').dumps(record, indent=2))
    temporary.replace(status_path)
    print(f"[{record['updated']}] {stage}: {state} {detail}", flush=True)
def run(args, stage='Setup', cwd=None, env=None, log_path=None):
    status(stage, detail='Starting: ' + ' '.join(map(str, args)))
    started = time.monotonic()
    process = subprocess.Popen([str(x) for x in args], cwd=cwd, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1,
                               start_new_session=True)
    done = threading.Event()
    def heartbeat():
        while not done.wait(30):
            print(f'[{stage}] Still running ({int(time.monotonic()-started)} seconds elapsed)...', flush=True)
    thread = threading.Thread(target=heartbeat, daemon=True)
    thread.start()
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            if log_path:
                with open(log_path, 'a') as log:
                    log.write(line)
        rc = process.wait()
        if rc:
            status(stage, 'failed', f'Exit code {rc}')
            raise subprocess.CalledProcessError(rc, args)
        status(stage, 'completed', f'{int(time.monotonic()-started)} seconds')
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGTERM)
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        status(stage, 'interrupted', 'Subprocess stopped')
        raise
    finally:
        done.set()
git_env = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1')
if not repo.exists():
    run(['git', 'clone', REPO_URL, repo], env=git_env, stage='Clone repository')
if not (repo / '.git').exists():
    raise RuntimeError(f'{repo} is not a Git checkout; inspect it before continuing.')
run(['git', '-C', repo, 'fetch', 'origin'], env=git_env)
if subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain', '--untracked-files=no'], text=True).strip():
    raise RuntimeError(f'{repo} has local changes; preserve them before continuing.')
run(['git', '-C', repo, 'checkout', '--detach', PROJECT_REF], env=git_env)
print('Code commit:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip(), flush=True)
if not mamba.exists():
    run(['bash', '-c', 'set -o pipefail; curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba'])
if not (env / 'bin/python').exists():
    run([mamba, 'create', '-y', '-p', env, '-c', 'conda-forge', 'python=3.10', 'pip=24.0'], stage='Create Python environment')
python = env / 'bin/python'
marker = env / '.dependencies-ready'
if not marker.exists():
    run([python, '-m', 'pip', 'install', '-r', repo / 'requirements.txt'], stage='Install dependencies')
    run([python, '-m', 'pip', 'install', '--no-deps', '-e', repo])
    run([python, '-c', 'import torch, torchaudio, pytorch_lightning, hydra, kenlm; assert torch.cuda.is_available(), "PyTorch cannot access GPU"'])
    marker.touch()
check = """import sys, yaml
from pathlib import Path
cfg = yaml.safe_load(Path(sys.argv[1]).read_text())
root = Path(sys.argv[2])
missing = [root / (s['session'] + '.hdf5') for split in ('train', 'val', 'test') for s in cfg['dataset'][split] if not (root / (s['session'] + '.hdf5')).is_file()]
if missing:
 print('Missing', len(missing), 'configured data files', flush=True)
 sys.exit(1)
"""
def data_ready():
    return subprocess.run([str(python), '-c', check, str(repo / 'config/user/single_user.yaml'), str(data)]).returncode == 0
if not data_ready():
    if not archive.exists():
        partial = Path(str(archive) + '.part')
        status('Download dataset', detail='Starting download to ' + str(partial))
        downloaded = 0
        last_report = 0
        try:
            with urllib.request.urlopen(url, timeout=60) as response, partial.open('wb') as dest:
                total = int(response.headers.get('Content-Length') or 0)
                while True:
                    chunk = response.read(1024 * 1024)
                    if not chunk:
                        break
                    dest.write(chunk)
                    downloaded += len(chunk)
                    now = time.monotonic()
                    if now - last_report >= 10:
                        detail = f'{downloaded / 1e9:.2f} GB'
                        if total:
                            detail += f' / {total / 1e9:.2f} GB ({100 * downloaded / total:.1f}%)'
                        status('Download dataset', detail=detail)
                        last_report = now
                if total and downloaded != total:
                    raise IOError(f'Incomplete download: {downloaded} of {total} bytes')
        except BaseException:
            status('Download dataset', 'interrupted or failed', f'{downloaded / 1e9:.2f} GB received')
            raise
        partial.replace(archive)
        status('Download dataset', 'completed', f'{downloaded / 1e9:.2f} GB')
    status('Extract dataset', detail='Checking archive entries; this can take several minutes')
    with tarfile.open(archive, 'r:gz') as tf:
        for member in tf.getmembers():
            target = (root / member.name).resolve()
            if not target.is_relative_to(root.resolve()) or member.issym() or member.islnk():
                raise RuntimeError(f'Unsafe archive entry: {member.name}')
        members = tf.getmembers()
        last_report = time.monotonic()
        for index, member in enumerate(members, 1):
            tf.extract(member, root)
            if time.monotonic() - last_report >= 10 or index == len(members):
                status('Extract dataset', detail=f'{index}/{len(members)} archive entries extracted')
                last_report = time.monotonic()
    status('Extract dataset', 'completed')
    if not data_ready():
        raise RuntimeError('Extracted dataset is missing required sessions; inspect /content.')
status('Dataset verification', 'completed', 'Required train, validation and test files found')
link = repo / 'data'
if link.is_symlink():
    if link.resolve() != data.resolve():
        raise RuntimeError(f'{link} points elsewhere; inspect it before continuing.')
elif link.exists():
    raise RuntimeError(f'{link} already exists; inspect it before continuing.')
else:
    link.symlink_to(data, target_is_directory=True)
run_dir = root / 'emg2qwerty-results' / time.strftime('%Y%m%d-%H%M%S')
run_dir.mkdir(parents=True, exist_ok=False)
command = [python, '-m', 'emg2qwerty.train', 'user=single_user', 'trainer.accelerator=gpu', 'trainer.devices=1', 'trainer.max_epochs=1', f'trainer.max_steps={MAX_STEPS}', f'batch_size={BATCH_SIZE}', 'num_workers=2', 'trainer.limit_val_batches=2', 'trainer.limit_test_batches=1', f'hydra.run.dir={run_dir}']
status('Training and evaluation', detail=f'Up to 1 epoch and {MAX_STEPS} training steps; results: {run_dir}')
run(command, cwd=repo, stage='Training and evaluation', log_path=run_dir / 'console.log')
import shutil
result_zip = shutil.make_archive(str(run_dir), 'zip', run_dir)
status('Results', 'completed', f'Archive: {result_zip}. Download before removing the server.')


## Check runtime status independently

Run the next cell by itself on the **same Colab server** after the active cell has stopped. It only inspects files, logs, and processes; it does not resume training. A single notebook kernel runs one cell at a time, so a status cell submitted while training runs will wait in the queue.


In [ ]:
# Read-only runtime status. Run this cell after stopping the active cell.
# A notebook kernel executes one cell at a time, so this cannot run concurrently
# with a training cell on the same kernel. It does not start or stop anything.
from pathlib import Path
import json, os, shutil, subprocess
root = Path('/content')
if not root.exists():
    raise RuntimeError('Select the same Colab runtime to inspect its files.')
print('=== Colab runtime status ===')
record = root / 'emg2qwerty-status.json'
if record.exists():
    print('Last recorded notebook status:', record.read_text())
    print('This is a last report, not proof the process is still running.')
else:
    print('No status record: this may be a run of the earlier notebook version.')
for path in [root / 'emg2qwerty', root / 'emg2qwerty-env/bin/python',
             root / 'emg2qwerty-env/.dependencies-ready',
             root / 'emg2qwerty-data-2021-08.tar.gz.part',
             root / 'emg2qwerty-data-2021-08.tar.gz', root / 'emg2qwerty-data-2021-08',
             root / 'emg2qwerty/data']:
    if path.is_file():
        print(f'{path}: exists, {path.stat().st_size / 1e9:.3f} GB')
    elif path.is_dir():
        print(f'{path}: directory exists')
    elif path.is_symlink():
        print(f'{path}: broken link -> {os.readlink(path)}')
    else:
        print(f'{path}: not found')
data = root / 'emg2qwerty-data-2021-08'
if data.is_dir():
    files = list(data.rglob('*.hdf5'))
    print(f'Extracted HDF5 files found: {len(files)} (file count alone does not prove dataset is complete)')
    print('metadata.csv:', 'found' if (data / 'metadata.csv').exists() else 'not found')
print('Disk free: %.2f GB' % (shutil.disk_usage(root).free / 1e9))
print('\n=== Relevant running processes ===')
r = subprocess.run(['ps', '-eo', 'pid,etime,args'], text=True, capture_output=True)
lines = [line for line in r.stdout.splitlines() if any(x in line for x in ('emg2qwerty.train', 'pip install', 'micromamba create', 'emg2qwerty-data-2021-08.tar.gz')) and 'ps -eo' not in line]
print('\n'.join(lines) if lines else 'None found (does not guarantee a clean stop).')
print('\n=== Result files and recent training log ===')
results = root / 'emg2qwerty-results'
if results.exists():
    logs = sorted(results.rglob('console.log'), key=lambda p: p.stat().st_mtime, reverse=True)
    checkpoints = list(results.rglob('*.ckpt'))
    archives = list(results.glob('*.zip'))
    print('Checkpoints:', *[str(p) for p in checkpoints], sep='\n  ')
    print('Result ZIPs:', *[str(p) for p in archives], sep='\n  ')
    if logs:
        print('Latest log:', logs[0])
        print('\n'.join(logs[0].read_text(errors='replace').splitlines()[-30:]))
    else:
        print('No training log found; training may not have started.')
else:
    print('No results directory found; training may not have started.')


## Save results and stop the GPU

Download the ZIP under `/content/emg2qwerty-results` using the Colab Contents view, or copy it to Google Drive. Verify the results are saved, then select **Colab → Remove Server**. A fresh runtime downloads and extracts the dataset again.
